# Qwen 50-invoice rerun on the shared OCR holdout

Benchmark Qwen3-0.6B base, the 0.6B LoRA adapter, and Qwen3-1.7B base on the same 50 cached OCR records. This run reuses the completed adapter and does not retrain. Results are saved under `/kaggle/working/invoice-llm-rerun-20261112/qwen`.


In [ ]:
from pathlib import Path
import csv, json, shutil, zipfile, random, os, time, gc, logging, traceback, unicodedata, statistics
import numpy as np

input_root = Path('/kaggle/input')
out=Path('/kaggle/working/invoice-llm-rerun-20261112/qwen')
out.mkdir(parents=True,exist_ok=True)
manifests = list(input_root.rglob('manifest.csv'))
if len(manifests) == 1:
    # Kaggle expands ZIP uploads into directories when mounting dataset inputs.
    data_root = manifests[0].parent
else:
    archives = list(input_root.rglob('invoice-synthetic-1000-sft.zip'))
    if len(archives) != 1:
        raise FileNotFoundError(f'Expected one manifest.csv or pilot ZIP; manifests={manifests}; archives={archives}; input_root={list(input_root.iterdir())}')
    data_parent = Path('/kaggle/working/invoice-synthetic-1000')
    shutil.rmtree(data_parent, ignore_errors=True)
    data_parent.mkdir(parents=True)
    with zipfile.ZipFile(archives[0]) as archive:
        archive.extractall(data_parent)
    data_root = data_parent / 'dataset'
manifest = list(csv.DictReader((data_root / 'manifest.csv').open(encoding='utf-8-sig')))
ground_truth = json.loads((data_root / 'ground_truth.json').read_text(encoding='utf-8'))
assert len(manifest) == len(ground_truth) == 1000
assert {s: sum(r['split'] == s for r in manifest) for s in ('train','validation','test')} == {'train':900,'validation':50,'test':50}
for i, row in enumerate(manifest):
    assert (data_root / 'data' / row['file']).is_file()
    assert ground_truth[i]['invoice_no'] == row['invoice_no']
print('Verified 1,000 labeled PDFs:', {s:sum(r['split']==s for r in manifest) for s in ('train','validation','test')})
print('Test cohort includes prior 50-sample pilot; no test invoice enters training.')


## 2. Install only the extraction-model dependencies

The benchmark reuses the previously saved Docling/VietOCR output for the exact 50 held-out invoices, so this run does not initialize the OCR stack.


In [ ]:
import subprocess, sys
repo = Path('/tmp/enterprise-knowledge-ai-agent')
shutil.rmtree(repo, ignore_errors=True)
subprocess.run(['git','clone','--depth','1','--branch','main','https://github.com/Lenhan231/enterprise-knowledge-ai-agent.git',str(repo)],check=True)
source_commit = subprocess.check_output(['git','-C',str(repo),'rev-parse','HEAD'],text=True).strip()
processor_sources=list(input_root.rglob('invoice_text_processor.py'))
if len(processor_sources)!=1: raise FileNotFoundError(f'Expected one bundled invoice_text_processor.py; found {processor_sources}')
shutil.copy2(processor_sources[0],repo/'src/document_processing/invoice_text_processor.py')
sys.path.insert(0,str(repo/'src'))
packages = ['transformers>=4.51.0,<5','accelerate==1.15.0','peft>=0.14,<1','safetensors']
subprocess.run([sys.executable,'-m','pip','install','--disable-pip-version-check',*packages],check=True)
subprocess.run([sys.executable,'-m','pip','uninstall','-y','torchao'],check=False)
subprocess.run([sys.executable,'-c','import torch; assert torch.cuda.is_available(); print(torch.cuda.get_device_name(0))'],check=True)


## 3. Build production-shaped supervised examples

Each invoice contributes a header task and a table task. The serialized evidence matches the production prompts and schema. Column headers and cell values are built from the labeled invoice; this training setup does not include OCR errors.

In [ ]:
sys.path.insert(0,str(repo/'src'))
from document_processing.invoice_text_processor import (
    SCHEMA, HEADER_SCHEMA, TABLE_SCHEMA, InvoiceFields,
    _extraction_jobs, _prepare_extraction_input, _parse_json_response, _merge_job_fields,
)

INSTRUCTIONS = (
    'Trích xuất hóa đơn tiếng Việt chỉ từ OCR EVIDENCE; không dùng kiến thức ngoài bằng chứng. '
    'Không bịa, đoán, chuẩn hóa, tự tính hoặc sửa dữ liệu OCR. Giá trị không nhìn thấy hoặc '
    'không chắc là null; danh sách items không có dòng hàng là [].\n'
    'OUTPUT phải là đúng một JSON object hợp lệ khớp SCHEMA: chỉ được dùng các key trong '
    'SCHEMA, không markdown, không code fence, không giải thích. Mọi giá trị lá khác null '
    'phải là JSON string, được chép từ đúng vùng bằng chứng. Giữ nguyên chữ số và dấu phân '
    'cách OCR; không xuất giá trị số dưới dạng JSON number hoặc biểu thức tính toán.\n'
)
HEADER_RULES = ('MODE: INVOICE FIELDS. Trích invoice_no, date, seller, buyer và các tổng có nhãn rõ '
    '(subtotal, tax_amount, total_amount) từ OCR EVIDENCE. Seller là bên phát hành; buyer là bên mua. '
    'Chép seller/buyer từ vùng có nhãn tương ứng; không lấy tiêu đề, watermark, ghi chú hoặc địa chỉ '
    'làm tên bên mua/bán. invoice_no phải là số hóa đơn được ghi rõ, không phải mã phiếu xuất kho. '
    'Chép ngày và số tổng đúng như OCR; không tính tổng hoặc thuế từ dòng hàng. Không trích items.\n')
TABLE_RULES = ('MODE: TABLE. Chỉ xuất key items theo SCHEMA; không xuất bất kỳ field tổng nào. Chép từng '
    'mặt hàng theo từng dòng và đúng theo tiêu đề cột trong bảng; tên/thứ tự cột có thể thay đổi. '
    'description lấy từ ô tên hàng, quantity từ ô số lượng, unit từ ô đơn vị, unit_price từ ô đơn giá, '
    'amount từ ô thành tiền. Không chuyển giá trị giữa cột, không tính toán, không thêm lời giải thích '
    'vào giá trị. Mọi giá trị phải là chuỗi JSON giữ nguyên OCR hoặc null nếu ô trống/không rõ; không '
    'dùng chuỗi rỗng. Với description, chép nguyên văn toàn bộ chữ trong ô tên hàng của dòng đó. '
    'Giữ thứ tự dòng; bỏ dòng tiêu đề và dòng tổng/thuế.\n')
COLUMNS = ['STT','Tên hàng hóa, dịch vụ','ĐVT','SL','Đơn giá','Thành tiền']

def make_training_pairs(label):
    evidence_lines = [
        {'element_type':'text','page_number':1,'docling_label':'section_header','text':'HÓA ĐƠN GIÁ TRỊ GIA TĂNG'},
        {'element_type':'text','page_number':1,'docling_label':'text','text':f"Ngày {label['date']}"},
        {'element_type':'text','page_number':1,'docling_label':'text','text':label['seller']},
        {'element_type':'text','page_number':1,'docling_label':'text','text':f"Tên đơn vị mua hàng: {label['buyer']}"},
        {'element_type':'text','page_number':1,'docling_label':'text','text':f"Số hóa đơn: {label['invoice_no']}"},
        {'element_type':'text','page_number':1,'docling_label':'text','text':f"Cộng tiền hàng hóa, dịch vụ: {label['subtotal']}"},
        {'element_type':'text','page_number':1,'docling_label':'text','text':f"Thuế GTGT (8%): {label['tax_amount']}"},
        {'element_type':'text','page_number':1,'docling_label':'text','text':f"Tổng tiền thanh toán: {label['total_amount']}"},
    ]
    header_prompt = (INSTRUCTIONS + HEADER_RULES + f'SCHEMA:\n{json.dumps(HEADER_SCHEMA,ensure_ascii=False)}\n'
        + f'OCR EVIDENCE:\n{json.dumps(evidence_lines,ensure_ascii=False)}')
    header_target = {k:label.get(k) for k in HEADER_SCHEMA}
    rows=[]
    for i,item in enumerate(label.get('items') or [],start=1):
        rows.append({'source_row':i,'cells':{'STT':str(i),'Tên hàng hóa, dịch vụ':item.get('description'),
            'ĐVT':item.get('unit'),'SL':item.get('quantity'),'Đơn giá':item.get('unit_price'),'Thành tiền':item.get('amount')}})
    table_evidence={'table_index':0,'page_number':1,'columns':COLUMNS,'rows':rows}
    table_prompt=(INSTRUCTIONS + TABLE_RULES + f'SCHEMA:\n{json.dumps(TABLE_SCHEMA,ensure_ascii=False)}\n'
        + f'OCR TABLE EVIDENCE:\n{json.dumps(table_evidence,ensure_ascii=False)}')
    table_target={'items':[{k:item.get(k) for k in ('description','quantity','unit','unit_price','amount')} for item in label.get('items') or []]}
    return [(header_prompt,header_target),(table_prompt,table_target)]

by_file={r['file']:(r,ground_truth[i]) for i,r in enumerate(manifest)}
RUN_TRAINING=False  # Adapter already trained and recovered; set True only for a deliberate retrain.
if RUN_TRAINING:
    train_pairs=[pair for r in manifest if r['split']=='train' for pair in make_training_pairs(by_file[r['file']][1])]
    val_pairs=[pair for r in manifest if r['split']=='validation' for pair in make_training_pairs(by_file[r['file']][1])]
    assert len(train_pairs)==1800 and len(val_pairs)==100
else:
    train_pairs=[]; val_pairs=[]
print('RUN_TRAINING:',RUN_TRAINING,'| test PDFs:',sum(r['split']=='test' for r in manifest))


## 4. Fine-tune Qwen3-0.6B with LoRA

Use a frozen 0.6B base with attention and MLP LoRA adapters. Prompt tokens are masked from loss; only the target JSON is trained.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

MODEL_06='Qwen/Qwen3-0.6B'
REV_06='c1899de289a04d12100db370d81485cdf75e47ca'
MODEL_17='Qwen/Qwen3-1.7B'
REV_17='70d244cc86ccca08cf5af4e1e306ecf908b1ad5e'
MAX_LENGTH=2048
if RUN_TRAINING:
    # Clean retrain path: no pre-trained adapter dataset is required.
    adapter_dir=Path('/kaggle/working/qwen3-0.6b-invoice-lora')
    training_seconds=None
else:
    # Benchmark-only path: load the previously trained adapter from a Kaggle input.
    adapter_configs=[p for p in input_root.rglob('adapter_config.json') if (p.parent/'adapter_model.safetensors').is_file()]
    if len(adapter_configs)!=1: raise FileNotFoundError(f'Expected one trained adapter input; found {adapter_configs}')
    adapter_dir=adapter_configs[0].parent
    training_seconds=29437.423
    shutil.copytree(adapter_dir,out/'adapter',dirs_exist_ok=True)
    print('Using recovered trained adapter:',adapter_dir)
    print('Training record: 3 epochs, 339 steps, best eval loss 1.0850759e-05; runtime seconds:',training_seconds)

if RUN_TRAINING:
    from datasets import Dataset
    from transformers import Trainer, TrainingArguments
    from peft import LoraConfig, get_peft_model
    tokenizer=AutoTokenizer.from_pretrained(MODEL_06,revision=REV_06)
    if tokenizer.pad_token is None: tokenizer.pad_token=tokenizer.eos_token
    def encode_pairs(pairs):
        encoded=[]
        for prompt,target in pairs:
            prompt_text=tokenizer.apply_chat_template([{'role':'user','content':prompt}],tokenize=False,add_generation_prompt=True,enable_thinking=False)
            full_text=tokenizer.apply_chat_template([{'role':'user','content':prompt},{'role':'assistant','content':json.dumps(target,ensure_ascii=False)}],tokenize=False,add_generation_prompt=False,enable_thinking=False)
            prompt_ids=tokenizer(prompt_text,add_special_tokens=False).input_ids
            full_ids=tokenizer(full_text,add_special_tokens=False).input_ids
            if len(full_ids)>MAX_LENGTH:
                target_len=max(1,len(full_ids)-len(prompt_ids))
                prompt_ids=prompt_ids[-(MAX_LENGTH-target_len):]
                full_ids=prompt_ids+full_ids[-target_len:]
            labels=[-100]*min(len(prompt_ids),len(full_ids))+full_ids[min(len(prompt_ids),len(full_ids)):]
            encoded.append({'input_ids':full_ids,'labels':labels})
        return Dataset.from_list(encoded)
    train_ds=encode_pairs(train_pairs); val_ds=encode_pairs(val_pairs)
    print('Tokenized train/val:',len(train_ds),len(val_ds),'max sequence:',max(len(x) for x in train_ds['input_ids']))
else:
    print('Skipping repeat training; reusing the adapter from the completed 900-invoice SFT run.')


In [ ]:
if RUN_TRAINING:
    from torch.nn.utils.rnn import pad_sequence

    class CausalOnlyCollator:
        def __call__(self,features):
            input_ids=[torch.tensor(x['input_ids'],dtype=torch.long) for x in features]
            labels=[torch.tensor(x['labels'],dtype=torch.long) for x in features]
            return {
                'input_ids':pad_sequence(input_ids,batch_first=True,padding_value=tokenizer.pad_token_id),
                'labels':pad_sequence(labels,batch_first=True,padding_value=-100),
                'attention_mask':pad_sequence([torch.ones_like(x) for x in input_ids],batch_first=True,padding_value=0),
            }

    base_model=AutoModelForCausalLM.from_pretrained(MODEL_06,revision=REV_06,torch_dtype=torch.float16)
    base_model.config.use_cache=False
    base_model.gradient_checkpointing_enable()
    base_model.enable_input_require_grads()
    base_model=get_peft_model(base_model,LoraConfig(r=16,lora_alpha=32,lora_dropout=0.05,bias='none',task_type='CAUSAL_LM',target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj']))
    base_model.print_trainable_parameters()
    # Keep the trainable model in memory for the Trainer cell below.
else:
    print('No training model loaded; GPU memory available for OCR.')


In [ ]:
if RUN_TRAINING:
    adapter_dir=Path('/kaggle/working/qwen3-0.6b-invoice-lora')
    training_args=TrainingArguments(
        output_dir='/kaggle/working/qwen3-0.6b-invoice-training',
        num_train_epochs=3,
        learning_rate=1e-4,
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=8,
        eval_strategy='epoch',
        save_strategy='epoch',
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model='eval_loss',
        greater_is_better=False,
        logging_steps=20,
        fp16=True,
        gradient_checkpointing=True,
        optim='adamw_torch',
        lr_scheduler_type='cosine',
        warmup_ratio=0.05,
        report_to='none',
        seed=20261108,
        data_seed=20261108,
        dataloader_num_workers=2,
    )
    trainer=Trainer(model=base_model,args=training_args,train_dataset=train_ds,eval_dataset=val_ds,data_collator=CausalOnlyCollator())
    train_start=time.perf_counter()
    train_result=trainer.train()
    training_seconds=time.perf_counter()-train_start
    trainer.model.save_pretrained(adapter_dir)
    tokenizer.save_pretrained(adapter_dir)
    print('Adapter saved:',adapter_dir,'training seconds:',round(training_seconds,1))
    print('Train summary:',train_result.metrics)
    del trainer
    gc.collect()
    torch.cuda.empty_cache()
else:
    print('Reusing recovered adapter; no model training in this run.')


## 5. Load cached OCR for the 50 held-out PDFs

All three models receive the same previously saved Docling/VietOCR elements and table cells from these exact held-out invoices.


In [ ]:
cache_files=list(input_root.rglob('holdout-ocr.json'))
if len(cache_files)!=1: raise FileNotFoundError(f'Expected one cached holdout-ocr.json; found {cache_files}')
ocr_cache=json.loads(cache_files[0].read_text(encoding='utf-8'))
expected_test={r['file']:by_file[r['file']][1] for r in manifest if r['split']=='test'}
if len(ocr_cache)!=50 or {r['file'] for r in ocr_cache}!=set(expected_test):
    raise ValueError('Cached OCR records do not match the 50-file held-out cohort')
for record in ocr_cache:
    if record['expected']!=expected_test[record['file']]:
        raise ValueError(f"Cached label mismatch for {record['file']}")
    record['ocr_status']='success'
    record['ocr_seconds']=None
print('Loaded OCR cache:',cache_files[0])
print('50/50 exact holdout files; elements:',sum(len(r['elements']) for r in ocr_cache),'tables:',sum(len(r['tables']) for r in ocr_cache))
print('OCR pipeline: Docling 2.129.0 + fine-tuned VietOCR, cached from the established 50-invoice pilot.')


## 6. Benchmark base 0.6B, fine-tuned 0.6B, and base 1.7B

Inference uses the repository production job builder and merge/parser functions. Greedy decoding, same prompts, same OCR cache, same schema, and same 50 holdout files.

In [ ]:
from peft import PeftModel

def load_benchmark_model(model_id,revision,adapter=None):
    tok=AutoTokenizer.from_pretrained(model_id,revision=revision)
    if tok.pad_token is None: tok.pad_token=tok.eos_token
    model=AutoModelForCausalLM.from_pretrained(model_id,revision=revision,torch_dtype=torch.float16).to('cuda')
    if adapter: model=PeftModel.from_pretrained(model,adapter)
    model.eval()
    return tok,model

def run_inference_case(tok,model,record):
    ocr_input=_prepare_extraction_input(record['elements'],record['tables'])
    jobs=_extraction_jobs(ocr_input)
    parsed_jobs=[]; raw=[]
    for job in jobs:
        chat=tok.apply_chat_template([{'role':'user','content':job['prompt']}],tokenize=False,add_generation_prompt=True,enable_thinking=False)
        inputs=tok(chat,return_tensors='pt').to(model.device)
        with torch.inference_mode():
            output=model.generate(**inputs,max_new_tokens=job['max_new_tokens'],do_sample=False,pad_token_id=tok.eos_token_id)
        response=tok.decode(output[0][inputs['input_ids'].shape[-1]:],skip_special_tokens=True).strip()
        parsed=_parse_json_response(response)
        raw.append({'source':job['source'],'response':response,'parsed':parsed})
        if parsed is not None: parsed_jobs.append((job,parsed))
    fields,conflicts=_merge_job_fields(parsed_jobs)
    return fields,conflicts,raw

model_specs=[('qwen3-0.6b-base',MODEL_06,REV_06,None),('qwen3-0.6b-lora',MODEL_06,REV_06,str(adapter_dir)),('qwen3-1.7b-base',MODEL_17,REV_17,None)]
all_outputs={}
for name,model_id,revision,adapter in model_specs:
    print('Loading',name,revision)
    tok,model=load_benchmark_model(model_id,revision,adapter)
    outputs=[]
    for i,record in enumerate(ocr_cache,1):
        if record['ocr_status']!='success':
            outputs.append({'file':record['file'],'expected':record['expected'],'predicted':{},'run_status':'ocr_failed','raw':[]})
            continue
        started=time.perf_counter()
        try:
            fields,conflicts,raw=run_inference_case(tok,model,record)
            outputs.append({'file':record['file'],'quality':record['quality'],'expected':record['expected'],'predicted':fields,'conflicts':conflicts,'raw':raw,'run_status':'success','inference_seconds':round(time.perf_counter()-started,2)})
        except Exception as exc:
            outputs.append({'file':record['file'],'quality':record['quality'],'expected':record['expected'],'predicted':{},'run_status':'failed','error':repr(exc),'raw':[]})
        (out/f'{name}-partial.json').write_text(json.dumps(outputs,ensure_ascii=False),encoding='utf-8')
        if i%5==0: print(name,i,'/',len(ocr_cache),'last_case_seconds',outputs[-1].get('inference_seconds'))
    all_outputs[name]=outputs
    del model,tok
    gc.collect(); torch.cuda.empty_cache()
    (out/f'{name}-results.json').write_text(json.dumps(outputs,ensure_ascii=False),encoding='utf-8')
    print('Done',name,'success',sum(x['run_status']=='success' for x in outputs))


## 7. Score and save the comparison

Exact match is normalized for Unicode whitespace/case; numeric fields compare digits only, matching the previous pilot metric. Item rows are matched by position. Both macro-per-invoice and pooled row accuracy are reported.

In [ ]:
HEADER_FIELDS=['invoice_no','date','seller','buyer','subtotal','tax_amount','total_amount']
ITEM_FIELDS=['description','quantity','unit','unit_price','amount']
NUMERIC_FIELDS={'subtotal','tax_amount','total_amount','quantity','unit_price','amount'}
def norm(value,numeric=False):
    if value is None:return ''
    text=unicodedata.normalize('NFC',' '.join(str(value).split())).casefold()
    return ''.join(ch for ch in text if ch.isdigit()) if numeric else text

def score_model(rows):
    scored=[]
    for row in rows:
        exp,pred=row['expected'],row.get('predicted') or {}
        try: InvoiceFields.model_validate(pred); schema=(row['run_status']=='success' and bool(row.get('raw')) and all(x.get('parsed') is not None for x in row['raw']))
        except Exception: schema=False
        exp_items=exp.get('items') or []; pred_items=pred.get('items') or []
        item_exact=[]
        field_matches={f:[] for f in ITEM_FIELDS}
        for i,ei in enumerate(exp_items):
            pi=pred_items[i] if i<len(pred_items) else {}
            checks={f:norm(ei.get(f),f in NUMERIC_FIELDS)==norm(pi.get(f),f in NUMERIC_FIELDS) for f in ITEM_FIELDS}
            item_exact.append(all(checks.values()))
            for f in ITEM_FIELDS:field_matches[f].append(checks[f])
        scored.append({'file':row['file'],'run_status':row['run_status'],'schema_valid':schema,'expected_item_count':len(exp_items),'predicted_item_count':len(pred_items),
          'item_count_exact':len(exp_items)==len(pred_items),'item_rows_exact':sum(item_exact),'item_rows_total':len(exp_items),
          'header_exact':{f:norm(exp.get(f),f in NUMERIC_FIELDS)==norm(pred.get(f),f in NUMERIC_FIELDS) for f in HEADER_FIELDS},
          'item_fields':{f:(sum(vals)/len(vals) if vals else None) for f,vals in field_matches.items()},
          'latency_seconds':row.get('inference_seconds')})
    total_items=sum(x['item_rows_total'] for x in scored)
    return {'sample_count':len(rows),'ocr_success_count':sum(r['ocr_status']=='success' for r in ocr_cache),
      'inference_success_count':sum(r['run_status']=='success' for r in rows),'schema_valid_rate':sum(x['schema_valid'] for x in scored)/len(scored),
      'header_exact_match_rates':{f:sum(x['header_exact'][f] for x in scored)/len(scored) for f in HEADER_FIELDS},
      'item_count_exact_rate':sum(x['item_count_exact'] for x in scored)/len(scored),
      'item_row_exact_accuracy_macro_invoice':sum((x['item_rows_exact']/x['item_rows_total'] if x['item_rows_total'] else float(x['predicted_item_count']==0)) for x in scored)/len(scored),
      'item_row_exact_accuracy_pooled':sum(x['item_rows_exact'] for x in scored)/total_items if total_items else 0,
      'item_field_accuracy_macro_invoice':{f:sum(x['item_fields'][f] or 0 for x in scored)/len(scored) for f in ITEM_FIELDS},
      'mean_inference_seconds':statistics.mean(x['latency_seconds'] for x in scored if x['latency_seconds'] is not None),
      'whole_invoice_exact_rate':sum(x['schema_valid'] and all(x['header_exact'].values()) and x['item_count_exact'] and x['item_rows_exact']==x['item_rows_total'] for x in scored)/len(scored)}

summary={name:score_model(rows) for name,rows in all_outputs.items()}
summary['experiment']={'train_invoices':900,'validation_invoices':50,'test_invoices':50,'training_seconds':round(training_seconds,1),'base_model_revision_0.6b':REV_06,'base_model_revision_1.7b':REV_17,'source_commit':source_commit,'ocr':'cached established pilot output: Docling 2.129.0 + fine-tuned VietOCR vgg_transformer','training_evidence':'synthetic OCR-like header/table records reconstructed from labels','test_cohort':'established invoice-pilot-50, held out from training'}
out=Path('/kaggle/working/invoice-llm-rerun-20261112/qwen');out.mkdir(exist_ok=True)
(out/'metrics.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding='utf-8')
(out/'results.json').write_text(json.dumps(all_outputs,ensure_ascii=False),encoding='utf-8')
for name,rows in all_outputs.items():
    (out/f'{name}-results.json').write_text(json.dumps(rows,ensure_ascii=False),encoding='utf-8')
print(json.dumps(summary,ensure_ascii=False,indent=2))


## 8. Export the adapter and benchmark artifacts

The adapter is saved separately from the base model. Metrics and per-invoice outputs are preserved for review.

In [ ]:
print('Benchmark artifacts:',out)
print('Recovered adapter copy:',out/'adapter')
